# Learning from chronological sequence data

A financial sequence of length $T$ is written as $X_{1:T}=(X_1,\ldots,X_T)$. Its joint distribution can be separated into conditional transition distributions:

$$
p(x_{1:T})=p(x_1)\prod_{t=2}^{T}p(x_t\mid x_{1:t-1}).
$$

Here, $X_t$ is the vector observed at time $t$; $x_t$ is one possible value of that vector; $x_{1:t-1}$ is the preceding history; $p(x_{1:T})$ is the joint density or mass function of the whole sequence. Each factor $p(x_t\mid x_{1:t-1})$ describes the next observation given the earlier observations. A useful generator must therefore represent both the marginal values and their order through time.

## Public Daily Market Data

The first feature is the daily continuously compounded return in percent:

$$
r_t=100\log\!\left(\frac{P_t}{P_{t-1}}\right).
$$

Here, $P_t$ and $P_{t-1}$ are consecutive valid NASDAQ Composite closing levels, and $r_t$ is the return from the preceding close to the current close. The factor $100$ expresses the result in percentage points.

The second feature is the sample standard deviation of the latest twenty returns:

$$
v_t=\left[\frac{1}{19}\sum_{j=0}^{19}\left(r_{t-j}-\bar r_t\right)^2\right]^{1/2},
\qquad
\bar r_t=\frac{1}{20}\sum_{j=0}^{19}r_{t-j}.
$$

Here, $v_t$ is the twenty-day return volatility; $\bar r_t$ is the mean of the same twenty returns; $j$ counts backward from the current date. The denominator is $19$ because the calculation uses the sample standard deviation of twenty observations.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI

assert RiskLabAI.__version__ == "3.2.0"

DATA_PATH = ensure_dataset(BOOK_ROOT, "fred_nasdaq")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "03_generative_financial_time_series"
TABLES = BOOK_ROOT / "outputs" / "tables" / "03_generative_financial_time_series"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "03_generative_financial_time_series"
ARTIFACTS = BOOK_ROOT / "outputs" / "artifacts" / "03_generative_financial_time_series"
for directory in [FIGURES, TABLES, FIGURE_DATA, ARTIFACTS]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)

from RiskLabAI.data.synthetic_data.timegan import chronological_timegan_data

In [2]:
expected_hash = "2E988948FFCB108DA4902DFEDAA5DF756E7E102DC523A99A8916A0533291F2D7"
assert DATA_PATH.is_file()
assert sha256(DATA_PATH) == expected_hash

raw = pd.read_csv(DATA_PATH)
raw["Date"] = pd.to_datetime(raw["observation_date"])
raw["NASDAQ Composite"] = pd.to_numeric(raw["NASDAQCOM"], errors="coerce")
valid = raw.dropna(subset=["NASDAQ Composite"]).sort_values("Date").reset_index(drop=True)
valid["Daily log return (%)"] = 100.0 * np.log(valid["NASDAQ Composite"] / valid["NASDAQ Composite"].shift(1))
valid["Twenty-day return volatility (%)"] = valid["Daily log return (%)"].rolling(20).std(ddof=1)
features = valid.dropna(subset=["Daily log return (%)", "Twenty-day return volatility (%)"]).reset_index(drop=True)
window_length = 20
training_end = int(np.floor(0.8 * len(features)))
split_date = features.loc[training_end, "Date"]

provenance = pd.DataFrame({
    "Item": ["Series", "Provider", "Source institution", "Frequency", "Local period", "File SHA256", "Download during run"],
    "Value": ["NASDAQ Composite (NASDAQCOM)", "Federal Reserve Economic Data", "Nasdaq, Inc.", "Daily close", f"{valid['Date'].min().date()} to {valid['Date'].max().date()}", expected_hash, "No"],
})
save_table(provenance, "data_provenance", "Public data and fixed local file identity.", "tab:chapter4-generative-data-provenance", fit=True)

split_summary = pd.DataFrame({
    "Segment": ["Training", "Held-out"],
    "First date": [features.loc[0, "Date"], features.loc[training_end, "Date"]],
    "Last date": [features.loc[training_end - 1, "Date"], features.loc[len(features) - 1, "Date"]],
    "Observations": [training_end, len(features) - training_end],
    "Windows": [training_end - window_length + 1, len(features) - training_end - window_length + 1],
})
save_table(split_summary, "chronological_split", "Chronological training and held-out segments.", "tab:chapter4-chronological-split")

summary_rows = []
for segment, part in [("Training", features.iloc[:training_end]), ("Held-out", features.iloc[training_end:])]:
    for feature in ["Daily log return (%)", "Twenty-day return volatility (%)"]:
        values = part[feature]
        summary_rows.append({
            "Segment": segment,
            "Feature": feature,
            "Mean": values.mean(),
            "Standard deviation": values.std(ddof=1),
            "Minimum": values.min(),
            "Median": values.median(),
            "Maximum": values.max(),
        })
feature_summary = pd.DataFrame(summary_rows)
save_table(feature_summary, "feature_summary", "Summary of the two public data features by chronological segment.", "tab:chapter4-feature-summary", fit=True)

public_series = valid[["Date", "NASDAQ Composite", "Daily log return (%)", "Twenty-day return volatility (%)"]].copy()
public_series["Segment"] = np.where(public_series["Date"] < split_date, "Training", "Held-out")
public_series.to_csv(FIGURE_DATA / "public_series.csv", index=False)
display(features.head())

,Item,Value
0,Series,NASDAQ Composite (NASDAQCOM)
1,Provider,Federal Reserve Economic Data
2,Source institution,"Nasdaq, Inc."
3,Frequency,Daily close
4,Local period,1994-01-03 to 2025-12-31
5,File SHA256,2E988948FFCB108DA4902DFEDAA5DF756E7E102DC523A9...
6,Download during run,No


,Segment,First date,Last date,Observations,Windows
0,Training,1994-01-31,2019-08-09,"6,428","6,409"
1,Held-out,2019-08-12,2025-12-31,"1,607","1,588"


,Segment,Feature,Mean,Standard deviation,Minimum,Median,Maximum
0,Training,Daily log return (%),0.035809,1.51531,-10.168411,0.110491,13.254645
1,Training,Twenty-day return volatility (%),1.296861,0.790706,0.342005,1.057281,5.505266
2,Held-out,Daily log return (%),0.066686,1.56119,-13.149154,0.142295,11.478439
3,Held-out,Twenty-day return volatility (%),1.374617,0.762798,0.444383,1.214157,6.016536


,observation_date,NASDAQCOM,Date,NASDAQ Composite,Daily log return (%),Twenty-day return volatility (%)
0,1994-01-31,800.47,1994-01-31,800.47,0.493426,0.351161
1,1994-02-01,797.24,1994-02-01,797.24,-0.404329,0.369004
2,1994-02-02,799.57,1994-02-02,799.57,0.291832,0.362094
3,1994-02-03,797.79,1994-02-03,797.79,-0.222868,0.368403
4,1994-02-04,777.28,1994-02-04,777.28,-2.604476,0.706119


In [3]:
figure, axes = plt.subplots(3, 1, figsize=(11.2, 9.2), sharex=True, constrained_layout=True)
axes[0].plot(valid["Date"], valid["NASDAQ Composite"], color=PALETTE["cyan"], linewidth=1.45)
axes[0].set_yscale("log")
axes[0].set_ylabel("Index level")
axes[0].set_title("Public NASDAQ Composite Series and Chronological Division")
axes[1].plot(features["Date"], features["Daily log return (%)"], color=PALETTE["amber"], linewidth=0.65)
axes[1].set_ylabel("Daily return (%)")
axes[2].plot(features["Date"], features["Twenty-day return volatility (%)"], color=PALETTE["green"], linewidth=1.0)
axes[2].set_ylabel("20-day volatility (%)")
axes[2].set_xlabel("Date")
for axis in axes:
    axis.axvline(split_date, color=PALETTE["rose"], linewidth=1.5, label="Held-out period begins")
    finish_axis(axis)
axes[0].legend(loc="upper left")
save_figure(figure, "public_series_split")
display(figure)
plt.close(figure)

<Figure size 1120x920 with 3 Axes>

## Chronological Separation Before Windowing

For a feature sequence $X_1,\ldots,X_n$ and window length $w$, the $i$th window is

$$
W_i=(X_i,X_{i+1},\ldots,X_{i+w-1}),
\qquad i=1,\ldots,n-w+1.
$$

Here, $X_t\in\mathbb R^d$ contains the $d$ features observed on date $t$; $w$ is the number of dates in one window; $W_i\in\mathbb R^{w\times d}$ is the window beginning at observation $i$. The chronological boundary is applied before this formula, so no training window contains a held-out date and no held-out window contains a training date.

Each feature is scaled with training data only:

$$
Z_{t,j}=\frac{X_{t,j}-a_j}{b_j},
\qquad
a_j=\min_{t\in\mathcal T}X_{t,j},
\qquad
b_j=\max_{t\in\mathcal T}X_{t,j}-a_j.
$$

Here, $j$ identifies a feature; $\mathcal T$ is the set of training dates; $a_j$ is the training minimum; $b_j$ is the training range; $Z_{t,j}$ is the scaled value. Held-out values are not used to find $a_j$ or $b_j$, and they are not clipped when they lie outside the training range.

In [4]:
feature_names = ["Daily log return (%)", "Twenty-day return volatility (%)"]
feature_array = features[feature_names].to_numpy(dtype=float)
prepared = chronological_timegan_data(feature_array, training_end=training_end, window_length=window_length)
training_windows = prepared["training"]
validation_windows = prepared["validation"]
minimum = prepared["minimum"]
scale = prepared["scale"]

assert training_windows.shape == (6409, 20, 2)
assert validation_windows.shape == (1588, 20, 2)
assert np.all((training_windows >= 0) & (training_windows <= 1))
assert np.allclose(training_windows[-1] * scale + minimum, feature_array[training_end - window_length:training_end])
assert np.allclose(validation_windows[0] * scale + minimum, feature_array[training_end:training_end + window_length])

np.savez_compressed(
    ARTIFACTS / "prepared_timegan_data.npz",
    training=training_windows,
    validation=validation_windows,
    minimum=minimum,
    scale=scale,
    feature_names=np.array(feature_names),
    dates=features["Date"].dt.strftime("%Y-%m-%d").to_numpy(),
    training_end=np.array([training_end]),
    window_length=np.array([window_length]),
)

scaling_rows = []
for index, feature in enumerate(feature_names):
    scaling_rows.append({
        "Feature": feature,
        "Training minimum": minimum[index],
        "Training maximum": minimum[index] + scale[index],
        "Training range": scale[index],
        "Held-out scaled minimum": validation_windows[:, :, index].min(),
        "Held-out scaled maximum": validation_windows[:, :, index].max(),
    })
scaling_table = pd.DataFrame(scaling_rows)
save_table(scaling_table, "training_scaling", "Training-only scaling and held-out ranges.", "tab:chapter4-training-scaling", fit=True)

scaled_full = (feature_array - minimum) / scale
scaled_series = pd.DataFrame({
    "Date": features["Date"],
    "Scaled daily return": scaled_full[:, 0],
    "Scaled twenty-day volatility": scaled_full[:, 1],
    "Segment": np.where(np.arange(len(features)) < training_end, "Training", "Held-out"),
})
scaled_series.to_csv(FIGURE_DATA / "scaled_feature_series.csv", index=False)

window_rows = []
for name, window in [("Last training window", training_windows[-1]), ("First held-out window", validation_windows[0])]:
    for day in range(window_length):
        for feature_index, feature in enumerate(feature_names):
            window_rows.append({"Window": name, "Relative day": day + 1, "Feature": feature, "Scaled value": window[day, feature_index]})
window_frame = pd.DataFrame(window_rows)
window_frame.to_csv(FIGURE_DATA / "chronological_window_examples.csv", index=False)
display(scaling_table)

,Feature,Training minimum,Training maximum,Training range,Held-out scaled minimum,Held-out scaled maximum
0,Daily log return (%),-10.168411,13.254645,23.423055,-0.127257,0.924169
1,Twenty-day return volatility (%),0.342005,5.505266,5.163261,0.019828,1.099021


,Feature,Training minimum,Training maximum,Training range,Held-out scaled minimum,Held-out scaled maximum
0,Daily log return (%),-10.168411,13.254645,23.423055,-0.127257,0.924169
1,Twenty-day return volatility (%),0.342005,5.505266,5.163261,0.019828,1.099021


In [5]:
figure, axes = plt.subplots(2, 1, figsize=(11.2, 7.3), constrained_layout=True)
axes[0].plot(scaled_series["Date"], scaled_series["Scaled daily return"], color=PALETTE["cyan"], linewidth=0.65, label="Daily return")
axes[0].plot(scaled_series["Date"], scaled_series["Scaled twenty-day volatility"], color=PALETTE["amber"], linewidth=0.9, label="Twenty-day volatility")
axes[0].axvline(split_date, color=PALETTE["rose"], linewidth=1.5)
axes[0].axhline(0, color=PALETTE["text"], linewidth=0.7)
axes[0].axhline(1, color=PALETTE["text"], linewidth=0.7)
axes[0].set_ylabel("Training-scaled value")
axes[0].set_title("Training-Only Scaling Leaves Held-Out Extremes Visible")
axes[0].legend(ncol=2, loc="upper left")
for window_name, color, style in [("Last training window", PALETTE["cyan"], "-"), ("First held-out window", PALETTE["amber"], "--")]:
    part = window_frame[(window_frame["Window"] == window_name) & (window_frame["Feature"] == "Daily log return (%)")]
    axes[1].plot(part["Relative day"], part["Scaled value"], color=color, linestyle=style, linewidth=2.0, marker="o", markersize=3.5, label=window_name)
axes[1].set_xlabel("Relative day in window")
axes[1].set_ylabel("Scaled daily return")
axes[1].set_title("No Window Crosses the Chronological Boundary")
axes[1].legend()
for axis in axes:
    finish_axis(axis)
save_figure(figure, "scaling_and_windows")
display(figure)
plt.close(figure)

<Figure size 1120x730 with 2 Axes>

## Tests

In [6]:
toy = np.column_stack([np.arange(20), np.ones(20)])
toy_result = chronological_timegan_data(toy, training_end=12, window_length=4)
assert np.allclose(toy_result["minimum"], [0, 1])
assert np.allclose(toy_result["scale"], [11, 1])
assert np.allclose(toy_result["training"][-1] * toy_result["scale"] + toy_result["minimum"], toy[8:12])
assert np.allclose(toy_result["validation"][0] * toy_result["scale"] + toy_result["minimum"], toy[12:16])
changed = toy.copy()
changed[12:] *= 100
altered = chronological_timegan_data(changed, training_end=12, window_length=4)
assert np.allclose(toy_result["training"], altered["training"])
assert np.allclose(toy_result["scale"], altered["scale"])

checks = pd.DataFrame({
    "Check": ["Public file identity", "Training windows", "Held-out windows", "Training values in unit interval", "Boundary reconstruction", "Held-out changes leave training scaling unchanged"],
    "Result": ["Pass", "Pass", "Pass", "Pass", "Pass", "Pass"],
})
save_table(checks, "chronology_checks", "Checks for public data identity, scaling, and chronological window construction.", "tab:chapter4-chronology-checks")

,Check,Result
0,Public file identity,Pass
1,Training windows,Pass
2,Held-out windows,Pass
3,Training values in unit interval,Pass
4,Boundary reconstruction,Pass
5,Held-out changes leave training scaling unchanged,Pass


,Check,Result
0,Public file identity,Pass
1,Training windows,Pass
2,Held-out windows,Pass
3,Training values in unit interval,Pass
4,Boundary reconstruction,Pass
5,Held-out changes leave training scaling unchanged,Pass
